<a href="https://colab.research.google.com/github/Poorvi-M/Heart-Disease-Prediction/blob/main/Heart_Disease_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Importing Data**

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("johnsmith88/heart-disease-dataset")
print("Path to dataset files:", path)

import pandas as pd
import os

files = os.listdir(path)
print("Files found:", files)
df = pd.read_csv(path + "/" + files[0])

!pip install ydata-profiling
from ydata_profiling import ProfileReport
import pandas as pd

ProfileReport(df).to_notebook_iframe()

**Data Exploration**

In [ ]:
print("Shape:", df.shape)
print("\nColumn names:\n", df.columns.tolist())
print("\nFirst 5 rows:")
df.describe()
df.value_counts
df.isnull().sum()
df.head(10)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
df.hist(bins=20, figsize=(15, 10))
plt.show()

In [ ]:
df[['chol', 'trestbps', 'age', 'thalach', 'oldpeak']].describe()

**Building a pipeline**

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X=df.drop('target', axis=1)
y=df['target']

pipeline = Pipeline([
    ('preprocessor', ColumnTransformer(
        transformers=[
            ('standard_scaler', StandardScaler(), ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']),
            ('ohe', OneHotEncoder(handle_unknown="ignore"), ['cp', 'restecg', 'thal'])
        ],
        remainder='passthrough'
    ))
])


**Selecting the best model**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV, train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = pipeline.named_steps['preprocessor']

models = {
    'Logistic Regression': LogisticRegression(),
    'Random Forest':       RandomForestClassifier(),
    'XGBoost':             XGBClassifier(),
    'SVM':                 SVC(),
    'Gradient Boosting':   GradientBoostingClassifier()
}


for name, model in models.items():
    model_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    scores = cross_val_score(model_pipeline, X_train, y_train, cv=5, scoring='recall')
    print(f"{name}: {scores.mean():.4f} \u00b1 {scores.std():.4f}")

param_grid_lr = {
    'model__C': [0.1, 1, 10],
    'model__penalty': ['l1', 'l2'],
    'model__solver': ['liblinear', 'saga'],
    'model__max_iter': [1000, 5000] # Increased max_iter for saga solver
}
param_grid_rf = {
    'model__n_estimators': [100, 200, 500],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 5, 10]
}
param_grid_xgb = {
    'model__n_estimators': [100, 200],
    'model__learning_rate': [0.01, 0.1],
    'model__max_depth': [3, 5]
}
param_grid_svm = {
    'model__C': [0.1, 1, 10, 100],
    'model__kernel': ['linear', 'rbf'],
    'model__gamma': ['scale', 'auto']
}
param_grid_gbc = {
    'model__n_estimators': [100, 200],
    'model__learning_rate': [0.01, 0.1],
    'model__max_depth': [3, 5]
}

In [ ]:

results = {}

for name, model, param_grid in [
    ('Logistic Regression', LogisticRegression(),     param_grid_lr),
    ('Random Forest',       RandomForestClassifier(), param_grid_rf),
    ('XGBoost',             XGBClassifier(),          param_grid_xgb)
]:
    best_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    grid_search = GridSearchCV(
        estimator=best_pipeline,
        param_grid=param_grid,
        cv=5,
        scoring=['recall','precision'],
        refit='recall',
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)

    results[name] = {
        'best_score'     : grid_search.best_score_,
        'best_params'    : grid_search.best_params_,
        'best_estimator' : grid_search.best_estimator_
    }

    print(f"\n{name}")
    print(f"  Best Params:    {results[name]['best_params']}")
    print(f"  Best CV Recall: {results[name]['best_score']:.4f}")


**Testing the model on test data and getting classification report**

In [ ]:
best_model = results['Random Forest']['best_estimator']
y_pred = best_model.predict(X_test)

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

**Random Forest Classifier turned out to be the best model with an accuracy of 99%**